# Baseline modell och logistisk regression modell

I denna notebook utvecklas en Baseline modell och en logistisk regression modell för sentimentklassificering av kundrecensioner från Amazon All_Beauty.

Målvariabeln är **sentiment**, som består av tre klasser:

* **negative** – betyg 1–2
* **neutral** – betyg 3
* **positive** – betyg 4–5

Texten representeras med hjälp av den TF-IDF-vektoriserare som skapades i det tidigare steget för textförbehandling.

## Modeller

I denna notebook utvecklas två modeller:

1. **Majoritetsbaslinje** – klassificerar alla recensioner som den vanligaste sentimentklassen.
2. **Logistisk regression** – en övervakad maskininlärningsmodell som tränas på TF-IDF-representationen av recensionerna.

Modellerna utvärderas på samma testdata för att se hur väl den logistiska regressionen presterar jämfört med en enkel baslinje.

> **Viktigt:** Kolumnen `rating` används inte som indata till modellerna eftersom sentimentvariabeln är direkt härledd från rating. Om `rating` skulle användas som en feature skulle det leda till target leakage.


In [1]:
import pandas as pd
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
reviews = pd.read_csv(
    "../data/processed/all_beauty_sample_100k.csv",
    keep_default_na=False
)

### Förbehandla recensionstexten

Här återskapas samma textförbehandling som användes i det tidigare steget. Detta behövs för att kunna använda den sparade TF-IDF-vektoriseraren på recensionstexten i denna notebook.

In [4]:
import re

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

reviews["clean_text"] = reviews["text"].apply(preprocess_text)

### Återskapa tränings- och testdata

I detta steg återskapas samma uppdelning i träningsdata (80 %) och testdata (20 %) som användes i textförbehandlingen. Samma `random_state` och stratifiering används för att säkerställa att samma datauppdelning återskapas och att fördelningen mellan sentimentklasserna bevaras.

Detta görs eftersom denna notebook använder den sparade TF-IDF-vektoriseraren och behöver skapa de tränings- och testdata som används för modelleringen.

In [5]:
from sklearn.model_selection import train_test_split

train_reviews, test_reviews = train_test_split(
    reviews,
    test_size=0.2,
    random_state=42,
    stratify=reviews["sentiment"]
)

print("Träningsdata:", len(train_reviews))
print("Testdata:", len(test_reviews))

Träningsdata: 80000
Testdata: 20000


### Load TF-IDF-vektoriseraren

Här läses den redan tränade TF-IDF-vektoriseraren från textförbehandlingssteget in. Genom att återanvända samma vektoriserare säkerställs att tränings- och testdata representeras på samma sätt.

In [6]:
tfidf = joblib.load(
    "../data/processed/tfidf_vectorizer.joblib"
)

print("Antal TF-IDF-features:", len(tfidf.get_feature_names_out()))

Antal TF-IDF-features: 10000


### Omvandla text till TF-IDF-funktioner

Här omvandlas den förbehandlade recensionstexten till numeriska TF-IDF-funktioner. Vektoriseraren tränas inte om, utan den tidigare skapade representationen används direkt.

In [7]:
X_train = tfidf.transform(train_reviews["clean_text"])
X_test = tfidf.transform(test_reviews["clean_text"])

y_train = train_reviews["sentiment"]
y_test = test_reviews["sentiment"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (80000, 10000)
X_test: (20000, 10000)
y_train: (80000,)
y_test: (20000,)


### Skapa majoritetsbaslinje

Som en enkel referensmodell används en majoritetsbaslinje. Modellen förutspår alltid den vanligaste sentimentklassen i träningsdata.

In [8]:
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

print("Majoritetsklass:", baseline.classes_[baseline.class_prior_.argmax()])
print("Baseline accuracy:", accuracy_score(y_test, baseline_predictions))

Majoritetsklass: positive
Baseline accuracy: 0.71305


### Tolkning av majoritetsbaslinjen

Majoritetsbaslinjen förutspår alltid den vanligaste sentimentklassen, vilket i detta dataset är **positive**. Modellen uppnår en accuracy på **71,3 %** på testdata.

Resultatet visar att en relativt hög accuracy kan uppnås även utan att analysera innehållet i recensionerna. Därför fungerar majoritetsbaslinjen som en enkel referenspunkt för den logistiska regressionsmodellen. För att visa att textbaserade TF-IDF-funktioner bidrar med faktisk prediktiv information bör den logistiska regressionen prestera bättre än denna baslinje.


### Träna logistisk regression

Här tränas en logistisk regressionsmodell på TF-IDF-representationen av recensionerna. Modellen lär sig sambandet mellan textens egenskaper och sentimentklassen.

In [9]:
logreg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logreg.fit(X_train, y_train)

print("Logistisk regression är tränad.")

Logistisk regression är tränad.


### Göra prediktioner

Efter att modellen har tränats används den för att förutsäga sentimentklassen för recensionerna i testdata.

In [10]:
logreg_predictions = logreg.predict(X_test)

print("Antal prediktioner:", len(logreg_predictions))

Antal prediktioner: 20000


### Beräkna accuracy

Här beräknas modellens accuracy på testdata. 

In [11]:
logreg_accuracy = accuracy_score(y_test, logreg_predictions)

print("Logistisk regression accuracy:", logreg_accuracy)

Logistisk regression accuracy: 0.863


### Utvärdera precision, recall och F1-score

Här beräknas precision, recall och F1-score för varje sentimentklass. Dessa mått ger en mer detaljerad bild av modellens prestanda än accuracy ensam.

In [12]:
print(classification_report(
    y_test,
    logreg_predictions,
    digits=3
))

              precision    recall  f1-score   support

    negative      0.787     0.787     0.787      4133
     neutral      0.451     0.159     0.235      1606
    positive      0.899     0.964     0.930     14261

    accuracy                          0.863     20000
   macro avg      0.712     0.637     0.651     20000
weighted avg      0.840     0.863     0.845     20000



### Utvärdering av logistisk regression

Den logistiska regressionsmodellen uppnådde en **accuracy på 86,3 %**, vilket är en tydlig förbättring jämfört med majoritetsbaslinjen på **71,3 %**.

Modellen presterar bäst för klassen **positive**, med ett F1-score på **0,930** och en recall på **0,964**. För klassen **negative** uppnås ett F1-score på **0,787**, vilket visar att modellen även kan identifiera negativa recensioner relativt väl.

Däremot har modellen betydligt svårare att identifiera **neutral**. Klassen har en precision på **0,451**, en recall på endast **0,159** och ett F1-score på **0,235**. Detta visar att accuracy ensam inte ger en fullständig bild av modellens prestanda, särskilt eftersom datasetet är obalanserat mellan sentimentklasserna.

Macro average för F1-score är **0,651**, medan weighted average är **0,845**. Skillnaden visar att den goda prestandan för den dominerande positiva klassen påverkar det viktade genomsnittet tydligt.


### Skapa Confusion Matrix

Här skapas en confusion matrix för att undersöka vilka sentimentklasser modellen klassificerar korrekt och vilka klasser som oftast förväxlas med varandra.

In [13]:
cm = confusion_matrix(
    y_test,
    logreg_predictions,
    labels=["negative", "neutral", "positive"]
)

print("Confusion matrix:")
print(cm)

Confusion matrix:
[[ 3251   141   741]
 [  544   255   807]
 [  338   169 13754]]


### Tolkning av Confusion matrix

Förväxlingsmatrisen visar att modellen klassificerar många recensioner korrekt för klasserna **negative** och **positive**. Av de negativa recensionerna klassificeras 3 251 korrekt, medan 13 754 av de positiva recensionerna klassificeras korrekt.

Den största svårigheten gäller klassen **neutral**. Av 1 606 faktiska neutrala recensioner klassificeras endast 255 korrekt. **544** klassificeras som negative och **807** som positive. Detta förklarar den låga recall på **15,9 %** för den neutrala klassen.

Resultatet visar att modellen har svårt att skilja neutrala recensioner från positiva och negativa recensioner. Den obalanserade klassfördelningen är en viktig faktor, eftersom neutral är den minst förekommande klassen i datasetet.


### Sammanställa modellresultat

Här sammanställs accuracy för majoritetsbaslinjen och den logistiska regressionen i en tabell. Tabellen gör det möjligt att tydligt jämföra resultatet med den enkla baslinjen.

In [14]:
results = pd.DataFrame({
    "Modell": [
        "Majoritetsbaslinje",
        "Logistisk regression"
    ],
    "Accuracy": [
        accuracy_score(y_test, baseline_predictions),
        accuracy_score(y_test, logreg_predictions)
    ]
})

results

,Modell,Accuracy
0,Majoritetsbaslinje,0.71305
1,Logistisk regression,0.86300
